# 04 - DuckDB Analytics

This notebook shows how to query HealthSynth outputs using DuckDB.

This is useful for:

- SQL practice
- analytics engineering
- dashboard prototyping
- data validation

In [ ]:
from healthsynth.generator import generate
import duckdb

In [ ]:
generate(
    config_path="../configs/profiles/oncology_training.yaml",
    output_dir="../output/duckdb_analytics",
    output_format="duckdb",
)

In [ ]:
con = duckdb.connect("../output/duckdb_analytics/healthsynth.duckdb")
con.execute("SHOW TABLES").fetchdf()

In [ ]:
con.execute("""
SELECT
    p.product_name,
    p.manufacturer,
    p.brand_type,
    SUM(rx.nrx) AS total_nrx,
    SUM(rx.trx) AS total_trx
FROM prescriptions rx
JOIN product p
    ON rx.product_id = p.product_id
GROUP BY
    p.product_name,
    p.manufacturer,
    p.brand_type
ORDER BY total_nrx DESC
""").fetchdf()

In [ ]:
con.execute("""
SELECT
    h.specialty,
    p.product_name,
    SUM(rx.nrx) AS total_nrx
FROM prescriptions rx
JOIN hcp_master h
    ON rx.hcp_id = h.hcp_id
JOIN product p
    ON rx.product_id = p.product_id
GROUP BY
    h.specialty,
    p.product_name
ORDER BY
    h.specialty,
    total_nrx DESC
""").fetchdf()

In [ ]:
con.execute("""
SELECT
    channel,
    COUNT(*) AS activities
FROM call_activity
GROUP BY channel
ORDER BY activities DESC
""").fetchdf()

In [ ]:
con.execute("""
SELECT
    DATE_TRUNC('month', rx_date::DATE) AS month,
    SUM(nrx) AS total_nrx
FROM prescriptions
GROUP BY month
ORDER BY month
""").fetchdf()

## Key Takeaway

HealthSynth can generate a queryable DuckDB database that behaves like a small commercial analytics warehouse.

This makes it useful for SQL practice, dashboard prototyping, and analytics engineering demos.